# B3 — W3: Fair Value vs Spread (Dual Axis)

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)


In [ ]:
wm = WINDOWS_META['W3']
rk = wm['result_key']
ts = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'timeseries.parquet')
ts.index = pd.to_datetime(ts.index)
ts['date'] = ts.index.date.astype(str)
daily = ts.groupby('date')[['fv','spread']].mean().reindex(wm['days'])

fig, ax1 = plt.subplots(figsize=(10, 5))
color_fv = '#3498db'
color_sp = '#e74c3c'
x = range(len(daily))
ax1.plot(x, daily['fv'].values, color=color_fv, marker='o', label='Fair Value (FV)')
ax1.set_ylabel('Fair Value (pts)', color=color_fv)
ax1.tick_params(axis='y', labelcolor=color_fv)
ax1.set_xticks(list(x))
ax1.set_xticklabels(wm['day_labels'])
ax1.set_xlabel('Roll Day')

ax2 = ax1.twinx()
ax2.plot(x, daily['spread'].values, color=color_sp, marker='s', ls='--', label='Spread')
ax2.set_ylabel('Calendar Spread (pts)', color=color_sp)
ax2.tick_params(axis='y', labelcolor=color_sp)

# shade the zone where FV > spread (carry compression)
fv_arr = daily['fv'].values
sp_arr = daily['spread'].values
for i in range(len(x)-1):
    if fv_arr[i] > sp_arr[i]:
        ax1.axvspan(x[i], x[i]+1, alpha=0.12, color='purple')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.set_title('W3 ESH5→ESM5: Fair Value Rising vs Spread Falling\n(shaded = FV > Spread squeeze zone)')
save_fig(fig, 'B3_w3_fv_vs_spread_dual_axis.png')
